# Full SOC Security Operations Copilot — End to End in One Notebook

This notebook is **self-contained**: every step of the project is defined here,
in order, with no imports from the project's `src/` folders. Run top to bottom
and you trace the whole system from data generation to the agentic block.

**The pipeline, in one line:**

```
P1-style data → fusion rules → risk scoring → RAG retrieval → LLM summary → governance-gated agent → human approval
```

**Sections**

0. Setup (paths, seed, imports)
1. Schema
2. Reference data (sites / zones / devices / users)
3. Surveillance events
4. Access logs
5. Fusion rules (4 detectors)
6. Risk scorer
7. Incidents (dedup + score + materialize)
8. Knowledge base (5 policy docs, inlined)
9. KB loader → Chroma vector store
10. Retriever (MMR + category routing)
11. Summarizer (Groq + citation guard, with stub fallback)
12. Policy gate (predicate evaluator — the reuse linchpin)
13. Governance state (dataclasses)
14. Audit log + memory + PII redactor
15. Governance nodes + routers + parsers
16. Graph builder (with the multi-step plan-loop FIX)
17. Domain tools (the 6 SOC actions)
18. Prompts + intake node
19. Copilot agent (wire it all together)
20. Run an incident end to end + inspect the audit trail


## 0. Setup

We run from the **repo root** so the cwd-relative data paths resolve the same
way the project's CLI does:
- reference CSVs → `data/reference/`
- synthetic parquet → `project_07_final_synthesis/data/synthetic/`
- KB + vector store → `project_07_final_synthesis/data/knowledge_base/`

`SEED = 42` is the single source of determinism — every generator draws from it.

In [1]:
from __future__ import annotations
import os, sys, json, re, csv, time, hashlib, sqlite3, inspect, argparse
from pathlib import Path
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass, field
from typing import Any, Callable, Literal, TypedDict

import numpy as np
import pandas as pd
import requests
import yaml

# --- Resolve the repo root and chdir there so cwd-relative paths work --------
# This notebook lives in project_07_final_synthesis/notebooks/. Repo root is two
# levels up. We walk up from the kernel cwd until we find the folder that
# contains project_07_final_synthesis/ - robust whether the kernel started in
# the notebook dir, the project dir, or the repo root.
NB_DIR = Path.cwd()
print("NB_DIR =", NB_DIR)
REPO_ROOT = next((p for p in [NB_DIR.resolve(), *NB_DIR.resolve().parents]
                  if (p / "project_07_final_synthesis").is_dir()), NB_DIR.parent)
PROJECT_DIR = REPO_ROOT / "project_07_final_synthesis"
os.chdir(REPO_ROOT)
print("cwd =", Path.cwd())

# --- The single source of determinism ----------------------------------------
SEED = 42
SCHEMA_VERSION = "p7-v0.1-vertical-slice"

# --- Data locations (cwd-relative after chdir) -------------------------------
REF_DIR           = Path("data/reference")
SYNTHETIC_DIR     = PROJECT_DIR / "data" / "synthetic"
KB_DIR            = PROJECT_DIR / "data" / "knowledge_base"
KB_JSONL          = KB_DIR / "knowledge_base.jsonl"
VECTOR_STORE_DIR  = KB_DIR / "vector_store"
AGENT_DATA_DIR    = PROJECT_DIR / "data"

for d in (REF_DIR, SYNTHETIC_DIR, KB_DIR, AGENT_DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

WRITE_CSV = True  # also write CSV mirrors of parquet for reviewers / Excel

NB_DIR = d:\AI_Master\Udacity\capstone_projects\project_07_final_synthesis\notebooks
cwd = D:\AI_Master\Udacity\capstone_projects


## 1. Schema

Schemas live in Python (dataclasses + column lists) and materialize to Parquet.
ID prefixes: `SITE-001`, `ZONE-003`, `DEV-014`, `USR-102`,
`EVT-000245`, `LOG-001992`, `INC-000041`, `KB-00012`.

In [2]:
@dataclass(frozen=True)
class Site:
    site_id: str; site_name: str; timezone: str

@dataclass(frozen=True)
class Zone:
    zone_id: str; site_id: str; zone_name: str; restricted: bool

@dataclass(frozen=True)
class Device:
    device_id: str; site_id: str; zone_id: str; device_type: str

@dataclass(frozen=True)
class User:
    user_id: str; site_id: str; role: str
    authorized_zones: tuple[str, ...]

SURVEILLANCE_EVENTS_COLS = [
    "event_id", "site_id", "zone_id", "device_id", "event_timestamp",
    "event_type", "confidence_score", "anomaly", "description",
]
ACCESS_LOGS_COLS = [
    "log_id", "site_id", "zone_id", "device_id", "log_timestamp",
    "user_id", "access_result", "reason",
]
INCIDENTS_COLS = [
    "incident_id", "site_id", "zone_id", "incident_start", "incident_end",
    "incident_type", "linked_event_ids", "linked_log_ids",
    "risk_score", "risk_band", "summary_text", "recommended_action",
    "citation_doc_ids", "human_review_required", "created_at",
]

def empty_surveillance(): return pd.DataFrame({c: pd.Series(dtype="object") for c in SURVEILLANCE_EVENTS_COLS})
def empty_access_logs():  return pd.DataFrame({c: pd.Series(dtype="object") for c in ACCESS_LOGS_COLS})
def empty_incidents():    return pd.DataFrame({c: pd.Series(dtype="object") for c in INCIDENTS_COLS})

# --- Parquet + CSV I/O --------------------------------------------------------
def write_parquet(df, name, out_dir=SYNTHETIC_DIR):
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{name}.parquet"
    df.to_parquet(path, index=False)
    if WRITE_CSV:
        df.to_csv(out_dir / f"{name}.csv", index=False, encoding="utf-8")
    return path

def read_parquet(name, in_dir=SYNTHETIC_DIR):
    path = Path(in_dir) / f"{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Parquet not found: {path}. Run the generator first.")
    return pd.read_parquet(path)

print("schema + I/O ready")

schema + I/O ready


# 📦 Generate data

Reference data, surveillance events, access logs.

## 2. Reference data — sites / zones / devices / users

Deterministic from `SEED`. We use the **scaled layout** (3 sites × 4 zones each
= 12 zones, 120 devices, 240 users) matching P1's naming convention
`SITE-NNN::ZONE-{A,B,C,D}`, where **ZONE-D is restricted** at every site.
That restricted zone is what the intrusion rule fires on.

In [3]:
N_SITES = 3
N_ZONES_PER_SITE = 4
N_DEVICES = 10 * N_SITES * N_ZONES_PER_SITE
N_USERS   = 20 * N_SITES * N_ZONES_PER_SITE
USE_SCALED_LAYOUT = N_ZONES_PER_SITE > 1
SCALED_ZONE_LETTERS = ["A", "B", "C", "D"]
SCALED_RESTRICTED_LETTER = "D"

def _site_id(i):  return f"SITE-{i+1:03d}"
def _zone_id(s, z):
    if USE_SCALED_LAYOUT:
        return f"{_site_id(s)}::ZONE-{SCALED_ZONE_LETTERS[z]}"
    return f"ZONE-{z+1:03d}"
def _device_id(i): return f"DEV-{i+1:03d}"
def _user_id(i):   return f"USR-{i+1:03d}"

def generate_sites(rng):
    return [{"site_id": _site_id(i), "site_name": f"Building-{i+1}", "timezone": "UTC"}
            for i in range(N_SITES)]

def generate_zones(rng):
    rows = []
    for s in range(N_SITES):
        for z in range(N_ZONES_PER_SITE):
            is_restricted = (SCALED_ZONE_LETTERS[z] == SCALED_RESTRICTED_LETTER) if USE_SCALED_LAYOUT else True
            rows.append({"zone_id": _zone_id(s, z), "site_id": _site_id(s),
                         "zone_name": f"Zone-{z+1}", "restricted": is_restricted})
    return rows

def generate_devices(rng):
    rows, counter = [], 0
    types, counts = ["camera", "badge_reader", "door"], [7, 2, 1]
    for s in range(N_SITES):
        for z in range(N_ZONES_PER_SITE):
            for dtype, n in zip(types, counts):
                for _ in range(n):
                    rows.append({"device_id": _device_id(counter), "site_id": _site_id(s),
                                 "zone_id": _zone_id(s, z), "device_type": dtype})
                    counter += 1
    return rows

def generate_users(rng):
    n_total = N_USERS
    roles = (["employee"]*int(n_total*0.6) + ["contractor"]*int(n_total*0.2)
             + ["cleaner"]*int(n_total*0.15) + ["security"]*int(n_total*0.05))
    assert len(roles) == n_total
    rows = []
    for i, role in enumerate(roles):
        site_idx, zone_idx = i % N_SITES, i % N_ZONES_PER_SITE
        auth_zones = [_zone_id(site_idx, zone_idx)]
        rows.append({"user_id": _user_id(i), "site_id": _site_id(i % N_SITES),
                     "role": role, "authorized_zones": ";".join(auth_zones)})
    return rows

def write_reference_csvs(out_dir=REF_DIR):
    out_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SEED)
    tables = {"sites.csv": generate_sites(rng), "zones.csv": generate_zones(rng),
              "devices.csv": generate_devices(rng), "users.csv": generate_users(rng)}
    for name, rows in tables.items():
        with (out_dir / name).open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    return tables

ref = write_reference_csvs()
sites   = pd.read_csv(REF_DIR / "sites.csv")
zones   = pd.read_csv(REF_DIR / "zones.csv")
devices = pd.read_csv(REF_DIR / "devices.csv")
users   = pd.read_csv(REF_DIR / "users.csv")
restricted = zones.loc[zones["restricted"], "zone_id"].tolist()
print(f"sites={len(sites)} zones={len(zones)} devices={len(devices)} users={len(users)}")
print("restricted zones:", restricted)

sites=3 zones=12 devices=120 users=240
restricted zones: ['SITE-001::ZONE-D', 'SITE-002::ZONE-D', 'SITE-003::ZONE-D']


## 3. Surveillance events

~50 camera detections. **Exactly 3 anomalies** (deterministic count, not a
per-row Bernoulli draw), concentrated in the restricted zone with high
confidence (Beta(8,2), mean ~0.8) so the intrusion rule trips naturally.

In [4]:
N_EVENTS = 50
N_ANOMALIES = 3
SIM_DURATION_HOURS = 24
CONFIDENCE_BETA_A, CONFIDENCE_BETA_B = 8.0, 2.0
EVENT_TYPES = ["person_detected", "person_detected", "person_detected",
               "vehicle_detected", "anomaly"]

def _event_id(i): return f"EVT-{i+1:06d}"

def generate_surveillance_events(rng, sites, zones, devices, n=N_EVENTS, n_anomalies=N_ANOMALIES):
    cameras = devices[devices["device_type"] == "camera"].reset_index(drop=True)
    restricted_zone_ids = set(zones.loc[zones["restricted"], "zone_id"])
    base_time = datetime(2026, 7, 1, 0, 0, 0, tzinfo=timezone.utc)
    anomaly_idx = set(rng.choice(n, size=n_anomalies, replace=False).tolist())
    rows = []
    for i in range(n):
        cam = cameras.iloc[i % len(cameras)]
        is_anomaly = i in anomaly_idx
        zone_id = next(iter(restricted_zone_ids)) if (is_anomaly and restricted_zone_ids) else cam["zone_id"]
        ts = base_time + timedelta(seconds=int(rng.uniform(0, SIM_DURATION_HOURS * 3600)))
        conf = float(np.clip(rng.beta(CONFIDENCE_BETA_A, CONFIDENCE_BETA_B), 0.0, 1.0))
        etype = "anomaly" if is_anomaly else str(rng.choice(EVENT_TYPES))
        desc = (f"Camera {cam['device_id']} flagged an anomaly in {zone_id}." if is_anomaly
                else f"Camera {cam['device_id']} detected a {etype.replace('_', ' ')} in {zone_id}.")
        rows.append({"event_id": _event_id(i), "site_id": cam["site_id"], "zone_id": zone_id,
                     "device_id": cam["device_id"], "event_timestamp": ts,
                     "event_type": etype, "confidence_score": round(conf, 3),
                     "anomaly": is_anomaly, "description": desc})
    df = pd.concat([empty_surveillance(), pd.DataFrame(rows)], ignore_index=True)
    df["confidence_score"] = df["confidence_score"].astype(float)
    df["anomaly"] = df["anomaly"].astype(bool)
    df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True)
    return df[SURVEILLANCE_EVENTS_COLS]

rng = np.random.default_rng(SEED)
events = generate_surveillance_events(rng, sites, zones, devices)
p = write_parquet(events, "surveillance_events")
an = int(events["anomaly"].sum())
print(f"wrote {p.name}  rows={len(events)} anomalies={an} ({an/len(events):.1%})")
events.head()

wrote surveillance_events.parquet  rows=50 anomalies=3 (6.0%)


,event_id,site_id,zone_id,device_id,event_timestamp,event_type,confidence_score,anomaly,description
0,EVT-000001,SITE-001,SITE-001::ZONE-A,DEV-001,2026-07-01 16:44:12+00:00,anomaly,0.651,False,Camera DEV-001 detected a anomaly in SITE-001:...
1,EVT-000002,SITE-001,SITE-001::ZONE-A,DEV-002,2026-07-01 03:04:29+00:00,person_detected,0.657,False,Camera DEV-002 detected a person detected in S...
2,EVT-000003,SITE-001,SITE-001::ZONE-A,DEV-003,2026-07-01 10:38:30+00:00,anomaly,0.886,False,Camera DEV-003 detected a anomaly in SITE-001:...
3,EVT-000004,SITE-001,SITE-001::ZONE-A,DEV-004,2026-07-01 15:09:35+00:00,vehicle_detected,0.654,False,Camera DEV-004 detected a vehicle detected in ...
4,EVT-000005,SITE-001,SITE-003::ZONE-D,DEV-005,2026-07-01 04:40:16+00:00,anomaly,0.804,True,Camera DEV-005 flagged an anomaly in SITE-003:...


## 4. Access logs

~200 badge-reader events, skewed ~87% granted / 8% denied / 5% invalid. Two
**deterministic injections** so the fusion rules fire every run:
- a **denial burst**: `USR-005` gets 4 denials in one zone within 1 hour (trips the "≥3 denials" rule), and
- one **tailgate** at a specific door with `reason=forced_door` (trips the tailgate rule).
Tailgate is never sampled randomly — only this one injection produces it.

In [5]:
N_LOGS = 200
GRANT_RATE, DENY_RATE, INVALID_RATE = 0.87, 0.08, 0.05
SIM_DURATION_HOURS = 24
DENIAL_BURST_USER, DENIAL_BURST_COUNT = "USR-005", 4
TAILGATE_DEVICE = "DEV-008"

def _log_id(i): return f"LOG-{i+1:06d}"

def _access_result(rng):
    r = rng.random()
    if r < GRANT_RATE: return "granted", "ok"
    if r < GRANT_RATE + DENY_RATE:
        return "denied", str(rng.choice(["expired", "wrong_zone", "revoked"]))
    return "invalid", "unknown_badge"

def generate_access_logs(rng, sites, zones, devices, users, n=N_LOGS):
    readers = devices[devices["device_type"] == "badge_reader"].reset_index(drop=True)
    base_time = datetime(2026, 7, 1, 0, 0, 0, tzinfo=timezone.utc)
    user_ids = users["user_id"].tolist()
    rows = []
    for i in range(n):
        reader = readers.iloc[i % len(readers)]
        result, reason = _access_result(rng)
        uid = DENIAL_BURST_USER if i < DENIAL_BURST_COUNT else str(rng.choice(user_ids))
        ts = base_time + timedelta(seconds=int(rng.uniform(0, SIM_DURATION_HOURS * 3600)))
        rows.append({"log_id": _log_id(i), "site_id": reader["site_id"], "zone_id": reader["zone_id"],
                     "device_id": reader["device_id"], "log_timestamp": ts,
                     "user_id": uid, "access_result": result, "reason": reason})
    df = pd.DataFrame(rows)
    # Override the first DENIAL_BURST_COUNT rows: clustered denials in one zone, same reader.
    burst_start = base_time + timedelta(hours=10)
    for k in range(DENIAL_BURST_COUNT):
        df.at[k, "log_timestamp"] = burst_start + timedelta(minutes=k * 5)
        df.at[k, "access_result"] = "denied"; df.at[k, "reason"] = "revoked"
        df.at[k, "device_id"] = readers.iloc[0]["device_id"]
        df.at[k, "zone_id"] = readers.iloc[0]["zone_id"]
    # One deterministic tailgate at the marked door.
    tg = min(N_LOGS - 1, DENIAL_BURST_COUNT + 2)
    df.at[tg, "device_id"] = TAILGATE_DEVICE
    df.at[tg, "access_result"] = "tailgate"; df.at[tg, "reason"] = "forced_door"
    df.at[tg, "log_timestamp"] = burst_start + timedelta(minutes=DENIAL_BURST_COUNT * 5 + 1)
    df = df.sort_values("log_timestamp").reset_index(drop=True)
    df["log_id"] = [f"LOG-{i+1:06d}" for i in range(len(df))]
    df["log_timestamp"] = pd.to_datetime(df["log_timestamp"], utc=True)
    return df[ACCESS_LOGS_COLS]

rng = np.random.default_rng(SEED)
logs = generate_access_logs(rng, sites, zones, devices, users)
p = write_parquet(logs, "access_logs")
print(f"wrote {p.name}  rows={len(logs)} "
      f"denied={int((logs['access_result']=='denied').sum())} "
      f"tailgate={int((logs['access_result']=='tailgate').sum())}")
logs.head()

wrote access_logs.parquet  rows=200 denied=18 tailgate=1


,log_id,site_id,zone_id,device_id,log_timestamp,user_id,access_result,reason
0,LOG-000001,SITE-003,SITE-003::ZONE-D,DEV-118,2026-07-01 00:25:41+00:00,USR-194,granted,ok
1,LOG-000002,SITE-002,SITE-002::ZONE-B,DEV-058,2026-07-01 00:30:27+00:00,USR-110,granted,ok
2,LOG-000003,SITE-002,SITE-002::ZONE-B,DEV-059,2026-07-01 00:31:07+00:00,USR-204,granted,ok
3,LOG-000004,SITE-001,SITE-001::ZONE-B,DEV-019,2026-07-01 00:44:22+00:00,USR-191,granted,ok
4,LOG-000005,SITE-003,SITE-003::ZONE-D,DEV-119,2026-07-01 00:51:55+00:00,USR-053,granted,ok


# 🧩 Fusion

Rule detectors, risk scoring, incident materialization.

## 5. Fusion rules — four detectors

Rules-first (P3 lesson: report what each rule fired on, not a black-box score).
Each `detect_*` returns candidate dicts; the risk scorer turns them into
final `INC-` rows. **ML is intentionally deferred.**

| Rule | Fires when |
|---|---|
| `intrusion_restricted` | anomaly in a restricted zone with confidence ≥ **0.85** |
| `repeated_denials` | ≥ **3** denials in the same zone within **60 min** |
| `cross_anomaly` | surveillance anomaly + unusual access in the same zone within **10 min** |
| `tailgate_door` | a tailgate log followed by `forced_door` activity within **10 min** |

The **0.85** here is the *confidence threshold* — a rule trigger, not a risk
score (the **80** risk gate comes later). Different stages, different scales.

In [6]:
RULE_INTRUSION_CONFIDENCE_MIN = 0.85
RULE_DENIAL_COUNT_MIN = 3
RULE_DENIAL_WINDOW_MIN = 60
RULE_CORRELATION_WINDOW_MIN = 10
RULE_TAILGATE_TO_DOOR_MIN = 10

RULE_NAMES = {
    "intrusion_restricted": "Restricted-Zone Intrusion",
    "repeated_denials": "Repeated Badge Denials",
    "cross_anomaly": "Surveillance + Access Anomaly Correlation",
    "tailgate_door": "Tailgating + Door Activity",
}

def _has(df, **cols): return all(c in df.columns for c in cols.values())

def detect_intrusion_restricted(events, zones):
    if not _has(events, event_id="event_id", zone_id="zone_id", event_type="event_type",
                confidence="confidence_score", anomaly="anomaly", timestamp="event_timestamp"):
        return []
    restricted = set(zones.loc[zones["restricted"], "zone_id"])
    ev = events[events["anomaly"].astype(bool)
                & events["zone_id"].isin(restricted)
                & (events["confidence_score"].astype(float) >= RULE_INTRUSION_CONFIDENCE_MIN)]
    return [{"incident_type": "suspected_unauthorized_entry", "rule": "intrusion_restricted",
             "linked_event_ids": [r.event_id], "linked_log_ids": [],
             "incident_start": r.event_timestamp, "incident_end": r.event_timestamp,
             "site_id": r.site_id, "zone_id": r.zone_id}
            for r in ev.itertuples(index=False)]

def detect_repeated_denials(logs, window_min=RULE_DENIAL_WINDOW_MIN, min_count=RULE_DENIAL_COUNT_MIN):
    if not _has(logs, log_id="log_id", zone_id="zone_id", timestamp="log_timestamp", result="access_result"):
        return []
    if logs.empty: return []
    denied = logs[logs["access_result"] == "denied"].sort_values("log_timestamp")
    if denied.empty: return []
    out, window = [], timedelta(minutes=window_min)
    for zone_id, group in denied.groupby("zone_id"):
        rows = list(group.itertuples(index=False)); n = len(rows)
        for i in range(n):
            win = [rows[i]]
            for j in range(i + 1, n):
                if (rows[j].log_timestamp - rows[i].log_timestamp) <= window:
                    win.append(rows[j])
                else: break
            if len(win) >= min_count:
                out.append({"incident_type": "repeated_badge_denials", "rule": "repeated_denials",
                            "linked_event_ids": [], "linked_log_ids": [r.log_id for r in win],
                            "incident_start": min(r.log_timestamp for r in win),
                            "incident_end": max(r.log_timestamp for r in win),
                            "site_id": rows[0].site_id, "zone_id": zone_id})
                break
    return out

def detect_cross_anomaly(events, logs, window_min=RULE_CORRELATION_WINDOW_MIN):
    if not _has(events, anomaly="anomaly", zone_id="zone_id", timestamp="event_timestamp"): return []
    if not _has(logs, result="access_result", zone_id="zone_id", timestamp="log_timestamp"): return []
    if events.empty or logs.empty: return []
    unusual = logs[logs["access_result"].isin(["denied", "invalid", "tailgate"])]
    if unusual.empty: return []
    out, window = [], timedelta(minutes=window_min)
    for ev in events[events["anomaly"].astype(bool)].itertuples(index=False):
        cands = unusual[(unusual["zone_id"] == ev.zone_id)
                        & unusual["log_timestamp"].between(ev.event_timestamp - window, ev.event_timestamp + window)]
        if cands.empty: continue
        c = cands.iloc[0]
        out.append({"incident_type": "cross_anomaly_correlation", "rule": "cross_anomaly",
                    "linked_event_ids": [ev.event_id], "linked_log_ids": [c["log_id"]],
                    "incident_start": min(ev.event_timestamp, c["log_timestamp"]),
                    "incident_end": max(ev.event_timestamp, c["log_timestamp"]),
                    "site_id": ev.site_id, "zone_id": ev.zone_id})
    return out

def detect_tailgate_door(events, logs, devices, window_min=RULE_TAILGATE_TO_DOOR_MIN):
    if logs.empty or events.empty: return []
    tails = logs[logs["access_result"] == "tailgate"]
    door_events = logs[logs["reason"] == "forced_door"]
    if tails.empty or door_events.empty: return []
    window, out = timedelta(minutes=window_min), []
    for t in tails.itertuples(index=False):
        near = door_events[(door_events["zone_id"] == t.zone_id)
                           & door_events["log_timestamp"].between(t.log_timestamp, t.log_timestamp + window)]
        if near.empty: continue
        nb = near.iloc[0]
        linked = list(dict.fromkeys([t.log_id, nb["log_id"]]))  # dedupe same row
        out.append({"incident_type": "tailgate_door_activity", "rule": "tailgate_door",
                    "linked_event_ids": [], "linked_log_ids": linked,
                    "incident_start": min(t.log_timestamp, nb["log_timestamp"]),
                    "incident_end": max(t.log_timestamp, nb["log_timestamp"]),
                    "site_id": t.site_id, "zone_id": t.zone_id})
    return out

def all_candidates(events, logs, zones, devices):
    c = []
    c.extend(detect_intrusion_restricted(events, zones))
    c.extend(detect_repeated_denials(logs))
    c.extend(detect_cross_anomaly(events, logs))
    c.extend(detect_tailgate_door(events, logs, devices))
    return c

raw = all_candidates(events, logs, zones, devices)
from collections import Counter
print("raw candidates:", len(raw), "by rule:", Counter(c["rule"] for c in raw))

raw candidates: 3 by rule: Counter({'intrusion_restricted': 1, 'repeated_denials': 1, 'tailgate_door': 1})


## 6. Risk scorer

Per-rule **base** + **size bonus** (more linked evidence) + **confidence bonus**
(surveillance rules), capped at **100**. Risk bands:

```
critical >= 80   (forces human review — this is the 80 gate)
high     >= 60
medium   >= 40
low      <  40
```

In [ ]:
RULE_BASE_RISK = {
    "intrusion_restricted": 70, 
    "repeated_denials": 55,
    "cross_anomaly": 60, 
    "tailgate_door": 65,
}

LINK_BONUS_PER, LINK_BONUS_CAP = 3, 15

CONFIDENCE_BONUS_CAP = 10

def _band(score):
    if score >= 80: return "critical"
    if score >= 60: return "high"
    if score >= 40: return "medium"
    return "low"

def score_candidate(candidate, events_lookup=None):
    rule = candidate["rule"]
    base = RULE_BASE_RISK.get(rule, 40)
    n_links = len(candidate.get("linked_event_ids", [])) + len(candidate.get("linked_log_ids", []))
    size_bonus = min(n_links * LINK_BONUS_PER, LINK_BONUS_CAP)
    conf_bonus = 0.0
    if events_lookup:
        confs = [events_lookup[e] for e in candidate.get("linked_event_ids", []) if e in events_lookup]
        if confs:
            conf_bonus = min((sum(confs)/len(confs)) * CONFIDENCE_BONUS_CAP, CONFIDENCE_BONUS_CAP)
    score = min(int(base + size_bonus + conf_bonus), 100)
    return {**candidate, "risk_score": score, "risk_band": _band(score)}

# quick demo on the raw candidates
events_lookup = dict(zip(events["event_id"], events["confidence_score"].astype(float)))
demo = [score_candidate(c, events_lookup) for c in raw[:6]]
for d in demo:
    print(f"{d['rule']:22s} score={d['risk_score']:3d} band={d['risk_band']:8s} links={len(d['linked_event_ids'])+len(d['linked_log_ids'])}")

intrusion_restricted   score= 82 band=critical links=1
repeated_denials       score= 67 band=high     links=4
tailgate_door          score= 68 band=high     links=1


## 7. Incidents — dedup + score + materialize

Deduplicate on `(rule, sorted linked ids)`, score each survivor, materialize to the
`INCIDENTS_COLS` schema, write `incidents.parquet`. The
`summary_text` / `recommended_action` / `citation_doc_ids` columns are left
**empty here** — the RAG+LLM step fills them later. Fusion stays
language-agnostic so it's testable without an LLM.

In [ ]:
def _dedup_key(c):
    return (c["rule"],
            tuple(sorted(c.get("linked_event_ids", []))),
            tuple(sorted(c.get("linked_log_ids", []))))

def _incident_id(i): 
    return f"INC-{i+1:06d}"

def build_incidents(events, logs, zones, devices) -> pd.DataFrame:
    raw = all_candidates(events, logs, zones, devices)
    seen, deduped = set(), []
    for c in raw:
        k = _dedup_key(c)
        if k in seen: continue
        seen.add(k); deduped.append(c)
    events_lookup = dict(zip(events["event_id"], events["confidence_score"].astype(float)))
    scored = [score_candidate(c, events_lookup) for c in deduped]
    now = datetime.now(timezone.utc)
    rows = []
    for i, c in enumerate(scored):
        rows.append({
            "incident_id": _incident_id(i), "site_id": c["site_id"], "zone_id": c["zone_id"],
            "incident_start": c["incident_start"], "incident_end": c["incident_end"],
            "incident_type": c["incident_type"],
            "linked_event_ids": ",".join(c.get("linked_event_ids", [])),
            "linked_log_ids": ",".join(c.get("linked_log_ids", [])),
            "risk_score": int(c["risk_score"]), "risk_band": c["risk_band"],
            "summary_text": "", "recommended_action": "", "citation_doc_ids": "",
            "human_review_required": c["risk_band"] == "critical",
            "created_at": now, "_rule": c["rule"],
        })
    df = pd.concat([empty_incidents(), pd.DataFrame(rows)], ignore_index=True)
    df["risk_score"] = df["risk_score"].astype(int)
    df["human_review_required"] = df["human_review_required"].astype(bool)
    for col in ("incident_start", "incident_end", "created_at"):
        df[col] = pd.to_datetime(df[col], utc=True)
    return df

incidents = build_incidents(events, logs, zones, devices)
p = write_parquet(incidents, "incidents")
by_band = incidents["risk_band"].value_counts().to_dict()
crit = int(incidents["human_review_required"].sum())
print(f"wrote {p.name}  rows={len(incidents)} by_band={by_band} critical={crit}")
incidents[["incident_id","incident_type","risk_score","risk_band","_rule","human_review_required"]]

wrote incidents.parquet  rows=3 by_band={'high': 2, 'critical': 1} critical=1


,incident_id,incident_type,risk_score,risk_band,_rule,human_review_required
0,INC-000001,suspected_unauthorized_entry,82,critical,intrusion_restricted,True
1,INC-000002,repeated_badge_denials,67,high,repeated_denials,False
2,INC-000003,tailgate_door_activity,68,high,tailgate_door,False


# 📚 RAG

Policy KB, Chroma vector store, MMR + category routing.

## 8. Knowledge base — 5 policy docs (inlined)

The KB is inlined here so the notebook is fully self-contained. Each doc has a
`doc_id`, `title`, `category`, `body`. The **category** is the 1:1 alignment
between the fusion layer's `incident_type` and the KB — it's what category
routing (Section 10) uses.

In [9]:
KB_DOCS = [
  {"doc_id": "KB-00001", "title": "Restricted-Zone Intrusion Response", "category": "intrusion",
   "body": "When a surveillance anomaly with confidence >= 0.85 occurs in a restricted zone, treat the event as a suspected unauthorized entry. First action: dispatch on-site security to the affected zone within 5 minutes. Second action: pull the camera feed covering the affected zone_id for the 10 minutes before and after the event_timestamp and archive the clip. Third action: check access_logs for any denied or invalid badge attempt in the same zone within 10 minutes of the event; if found, the case is escalated to critical and a human reviewer must verify before any physical response. Do not lock doors automatically - door-lock actions must be approved by a human reviewer. Retention: the incident record and all linked evidence must be retained for 90 days minimum per the site retention policy."},
  {"doc_id": "KB-00002", "title": "Repeated Badge Denials Response", "category": "denials",
   "body": "Three or more denied or invalid badge attempts in the same zone within a 60-minute window is a credential-attack indicator. First action: review the user_id history across all sites for the prior 7 days. Second action: if the same user_id has denials in more than one site or zone in that window, suspend the badge pending identity verification. Third action: notify the duty manager via the standard channel. Do not auto-disable the user account - disable actions require human approval. Retention: 90 days. Privacy: the user_id is internal; do not include the badge holder's name in any external-facing summary."},
  {"doc_id": "KB-00003", "title": "Surveillance + Access Anomaly Correlation", "category": "correlation",
   "body": "When a surveillance anomaly (any confidence) and an unusual access event (denied, invalid, or tailgate) occur in the same zone within 10 minutes, treat them as a single correlated incident. The combined signal is stronger than either alone. First action: dispatch security if the access event is denied or tailgate. Second action: pull the camera clip covering the anomaly event. Third action: if both signals are in a restricted zone, escalate to critical. Always cite this doc when recommending a correlated response."},
  {"doc_id": "KB-00004", "title": "Tailgating + Door Activity Response", "category": "tailgate",
   "body": "A tailgate log (one badge grant with multiple persons passing) followed by door-sensor activity (forced_door reason) within 10 minutes in the same zone indicates a possible physical breach. First action: dispatch security. Second action: hold the door in locked state pending on-site arrival; do not auto-unlock. Third action: review the camera feed for the badge_id used in the tailgate. If the badge_id belongs to a user with a different role (e.g. cleaner in a server room) the case is escalated to critical. Human approval is required before any door-state change."},
  {"doc_id": "KB-00005", "title": "Privacy, PII, and Access-Log Retention", "category": "privacy",
   "body": "Access logs may contain user_id (internal) but must NOT include badge holder names, contact details, or biometric data in any external-facing summary. The summarizer must redact any user_name, email, or phone fields if present. Retention: access logs are retained for 90 days; surveillance event metadata (no raw video) for 180 days; incident records for 1 year. Human review is mandatory for any incident with risk_band = critical. The copilot never auto-resolves a critical incident."},
]

# Write the JSONL the loader reads (keeps the on-disk format identical to the project).
with KB_JSONL.open("w", encoding="utf-8") as f:
    for d in KB_DOCS:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")
print(f"wrote {KB_JSONL}  docs={len(KB_DOCS)}")
for d in KB_DOCS:
    print(f"  {d['doc_id']} [{d['category']:11s}] {d['title']}")

wrote D:\AI_Master\Udacity\capstone_projects\project_07_final_synthesis\data\knowledge_base\knowledge_base.jsonl  docs=5
  KB-00001 [intrusion  ] Restricted-Zone Intrusion Response
  KB-00002 [denials    ] Repeated Badge Denials Response
  KB-00003 [correlation] Surveillance + Access Anomaly Correlation
  KB-00004 [tailgate   ] Tailgating + Door Activity Response
  KB-00005 [privacy    ] Privacy, PII, and Access-Log Retention


## 9. KB loader → Chroma vector store

**Native `chromadb` + `sentence-transformers`** — no LangChain wrappers.
A 5-doc KB with one embedding model has nothing to swap, so the LangChain
`VectorStore` abstraction would be an interface with a single implementation.
We embed `title + body` (the title carries signal for short docs) and store
with metadata `doc_id / title / category` so the retriever can return ids for
citation without re-parsing.

In [10]:
import chromadb
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
COLLECTION_NAME = "p7_policy_kb"

def load_kb_docs(path=KB_JSONL):
    docs, required = [], {"doc_id", "title", "category", "body"}
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            missing = required - row.keys()
            if missing: raise ValueError(f"{path} missing keys {missing}")
            docs.append(row)
    return docs

def _embed_text(doc):  # title carries signal for short policy docs
    return f"{doc['title']}\n{doc['body']}"

def build_vector_store(kb_path=KB_JSONL, store_dir=VECTOR_STORE_DIR,
                       collection_name=COLLECTION_NAME, model_name=EMBED_MODEL_NAME):
    docs = load_kb_docs(kb_path)
    store_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(store_dir))
    try: client.delete_collection(collection_name)
    except Exception: pass
    collection = client.get_or_create_collection(collection_name, metadata={"hnsw:space": "cosine"})
    model = SentenceTransformer(model_name)
    texts = [_embed_text(d) for d in docs]
    embeddings = model.encode(texts, normalize_embeddings=True).tolist()
    collection.add(ids=[d["doc_id"] for d in docs], documents=texts,
                   metadatas=[{"doc_id": d["doc_id"], "title": d["title"], "category": d["category"]} for d in docs],
                   embeddings=embeddings)
    return len(docs), store_dir

n, store_dir = build_vector_store()
print(f"built {COLLECTION_NAME}: {n} docs -> {store_dir}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

built p7_policy_kb: 5 docs -> D:\AI_Master\Udacity\capstone_projects\project_07_final_synthesis\data\knowledge_base\vector_store


## 10. Retriever — MMR with category routing

**Why MMR (Maximal Marginal Relevance)?** Plain cosine returns the *k* most
similar docs, which are often near-duplicates. MMR re-ranks so each of the k=3
docs is relevant to the query **but not a repeat of the others**:

```
score = λ · relevance(query, doc) − (1−λ) · redundancy(doc, already-picked)
```

λ = 0.5 → equal weight to relevance and diversity.

**Why category routing?** The KB docs share heavy vocabulary (badge, human
review, retention), so pure semantic similarity mis-ranks. But the fusion
layer already *knows* the `incident_type`, so we route on it: the matching
category gets a **+0.3** bonus before MMR, guaranteeing the right policy ranks
first. MMR then fills the rest with diverse neighbors. The returned `score` is
the raw cosine (pre-bonus) so transparency is preserved.

In [11]:
TOP_K = 3
MMR_POOL_N = 10
MMR_LAMBDA = 0.5
CATEGORY_BONUS = 0.3

# incident_type -> KB category (the 1:1 alignment between rule layer and KB)
INCIDENT_TYPE_TO_CATEGORY = {
    "suspected_unauthorized_entry": "intrusion",
    "repeated_badge_denials": "denials",
    "cross_anomaly_correlation": "correlation",
    "tailgate_door_activity": "tailgate",
}

_EMBED_MODEL_CACHE = None  # lazy singleton: ~3s/150MB load once per process

def _cos(a, b):
    a = a / (np.linalg.norm(a) + 1e-12)
    b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return b @ a  # (m,)

def _mmr_rerank(rel_sim, cand_embs, k, lam=MMR_LAMBDA):
    # Greedy MMR. O(m*k) - fine at m<=10; for a large KB cap m with ANN first.
    m = len(rel_sim)
    if m == 0: return []
    k = min(k, m)
    selected, remaining = [], list(range(m))
    max_sim_to_sel = np.full(m, -np.inf)
    while len(selected) < k and remaining:
        best_idx, best_score = None, -np.inf
        for i in remaining:
            diversity = 0.0 if not selected else max_sim_to_sel[i]
            score = lam * rel_sim[i] - (1 - lam) * diversity
            if score > best_score: best_score, best_idx = score, i
        selected.append(best_idx); remaining.remove(best_idx)
        new_sim = _cos(cand_embs[best_idx], cand_embs).ravel()
        max_sim_to_sel = np.maximum(max_sim_to_sel, new_sim)
    return selected

def retrieve(query, k=TOP_K, pool_n=MMR_POOL_N, lam=MMR_LAMBDA,
             category_hint=None, store_dir=VECTOR_STORE_DIR,
             collection_name=COLLECTION_NAME, model_name=EMBED_MODEL_NAME):
    client = chromadb.PersistentClient(path=str(store_dir))
    try: collection = client.get_collection(collection_name)
    except Exception: return []
    pool_n = min(pool_n, collection.count())
    if pool_n == 0: return []
    pool = collection.query(query_texts=[query], n_results=pool_n)
    ids, docs, metas = pool["ids"][0], pool["documents"][0], pool["metadatas"][0]
    if not ids: return []
    emb_rows = collection.get(ids=ids, include=["embeddings"])
    cand_embs = np.asarray(emb_rows["embeddings"], dtype=np.float32)
    global _EMBED_MODEL_CACHE
    if _EMBED_MODEL_CACHE is None or _EMBED_MODEL_CACHE[0] != model_name:
        _EMBED_MODEL_CACHE = (model_name, SentenceTransformer(model_name))
    model = _EMBED_MODEL_CACHE[1]
    q_emb = model.encode([query], normalize_embeddings=True)[0].astype(np.float32)
    rel_sim = _cos(q_emb, cand_embs)
    # category bonus applied to MMR ordering only; rel_sim left untouched (transparency)
    if category_hint:
        mmr_sim = rel_sim.copy()
        for i, m in enumerate(metas):
            if m.get("category") == category_hint: mmr_sim[i] += CATEGORY_BONUS
    else:
        mmr_sim = rel_sim
    order = _mmr_rerank(mmr_sim, cand_embs, k=k, lam=lam)
    return [{"doc_id": metas[idx]["doc_id"], "title": metas[idx]["title"],
             "category": metas[idx]["category"], "body": docs[idx],
             "score": float(rel_sim[idx])} for idx in order]

def retrieve_for_incident(incident_type, query, k=TOP_K, **kwargs):
    hint = INCIDENT_TYPE_TO_CATEGORY.get(incident_type)
    return retrieve(query, k=k, category_hint=hint, **kwargs)

# --- demo: route on a tailgate incident and watch the right doc rank first ---
res = retrieve_for_incident("tailgate_door_activity",
                            "tailgate followed by forced door activity in a restricted zone", k=3)
print("routed on incident_type=tailgate_door_activity ->")
for r in res:
    print(f"  [{r['doc_id']}] ({r['category']:11s}) score={r['score']:.3f}  {r['title']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

routed on incident_type=tailgate_door_activity ->
  [KB-00004] (tailgate   ) score=0.504  Tailgating + Door Activity Response
  [KB-00002] (denials    ) score=0.691  Repeated Badge Denials Response
  [KB-00003] (correlation) score=0.494  Surveillance + Access Anomaly Correlation


# ✍️ Summarizer

Groq LLM with a citation guard (stub fallback).

## 11. Summarizer — Groq LLM with a citation guard

Calls Groq `llama-3.1-8b-instant` (free tier) via plain `requests` — **no
OpenAI SDK**. The guard is the point: a free 8B model can invent a `KB-99999`,
so we validate every cited id against the retrieved set.

**Citation guard flow:**
1. Parse `KB-XXXXX` ids out of the model's summary + action + citations field.
2. Keep only ids that are in the retrieved set; drop the rest.
3. If zero valid ids survive → re-prompt **once** with a stricter instruction
   that lists the allowed ids explicitly.
4. If still zero → mark the summary `needs_review: no valid citation`.

If `GROQ_API_KEY` is not set, the cell runs in **stub mode** and writes a
deterministic placeholder summary citing the top retrieved doc — so the notebook
is fully runnable without a key. `--llm` mode (real Groq) is opt-in.

In [12]:
from dotenv import load_dotenv
load_dotenv(PROJECT_DIR / ".env")

GROQ_URL = os.environ.get("GROQ_BASE_URL", "https://api.groq.com/openai/v1") + "/chat/completions"
GROQ_MODEL = os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant")
GROQ_TIMEOUT = 60
GROQ_MAX_RETRIES = 3
KB_ID_RE = re.compile(r"KB-\d{5}")
HAS_GROQ_KEY = bool(os.environ.get("GROQ_API_KEY"))

def _api_key():
    key = os.environ.get("GROQ_API_KEY")
    if not key:
        raise RuntimeError("GROQ_API_KEY not set (stub mode will be used instead).")
    return key

def _chat(messages, temperature=0.2):
    headers = {"Authorization": f"Bearer {_api_key()}", "Content-Type": "application/json"}
    body = {"model": GROQ_MODEL, "messages": messages, "temperature": temperature}
    last = None
    for attempt in range(GROQ_MAX_RETRIES):
        try:
            resp = requests.post(GROQ_URL, headers=headers, json=body, timeout=GROQ_TIMEOUT)
        except requests.RequestException as e:
            last = e; time.sleep(2 ** attempt); continue
        if resp.status_code == 200:
            return resp.json()["choices"][0]["message"]["content"]
        if resp.status_code in (429, 500, 502, 503, 504):
            last = RuntimeError(f"Groq {resp.status_code}: {resp.text[:200]}")
            time.sleep(2 ** attempt); continue
        raise RuntimeError(f"Groq {resp.status_code}: {resp.text[:300]}")
    raise RuntimeError(f"Groq failed after {GROQ_MAX_RETRIES} retries: {last}")

def _build_prompt(incident, docs):
    allowed = ", ".join(d["doc_id"] for d in docs)
    policy_block = "\n\n".join(f"[{d['doc_id']}] {d['title']} ({d['category']})\n{d['body']}" for d in docs)
    system = ("You are a Security Operations Center copilot. You write concise, analyst-facing "
              "incident summaries grounded in the provided policy docs. Every claim about procedure "
              f"MUST cite a doc_id from the allowed set ({allowed}). Do NOT invent or cite any doc_id "
              "not in that set. Output STRICTLY as JSON with keys: summary (2-3 sentences), "
              "recommended_action (one concrete next step), citations (list of doc_id strings).")
    user = (f"Incident {incident['incident_id']} ({incident['incident_type']}), "
            f"risk_band={incident['risk_band']}, risk_score={incident['risk_score']}, "
            f"zone={incident['zone_id']}.\n"
            f"Linked events: {incident['linked_event_ids'] or 'none'}; "
            f"linked logs: {incident['linked_log_ids'] or 'none'}.\n\n"
            f"Relevant policies:\n{policy_block}\n\nWrite the JSON now.")
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def _extract_citations(text):
    return list(dict.fromkeys(KB_ID_RE.findall(text)))

def _parse_json_response(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except json.JSONDecodeError: pass
    return {"summary": text.strip(), "recommended_action": "", "citations": _extract_citations(text)}

def _summarize_once(incident, docs):
    parsed = _parse_json_response(_chat(_build_prompt(incident, docs)))
    cited = set(_extract_citations(parsed.get("summary", "")))
    cited |= set(_extract_citations(parsed.get("recommended_action", "")))
    cited |= set(parsed.get("citations", []) or [])
    parsed["citations"] = list(dict.fromkeys(cited))
    return parsed

def _stub_summary(incident, docs):
    # Deterministic fallback when no GROQ_API_KEY: cite the top retrieved doc.
    top = docs[0] if docs else None
    cite = top["doc_id"] if top else ""
    summ = (f"[stub] {incident['incident_type']} in {incident['zone_id']} "
            f"(risk_band={incident['risk_band']}, score={incident['risk_score']}).")
    action = (f"[stub] Follow {top['title']} ({cite})." if top else "[stub] No policy retrieved.")
    return {"summary": summ, "recommended_action": action, "citations": [cite] if cite else []}

def summarize_incident(incident):
    docs = retrieve_for_incident(incident["incident_type"],
                                 f"{incident['incident_type']} in zone {incident['zone_id']}; "
                                 f"events {incident['linked_event_ids']}; logs {incident['linked_log_ids']}", k=3)
    valid_ids = {d["doc_id"] for d in docs}
    def _validate(parsed): return [c for c in parsed.get("citations", []) if c in valid_ids]

    if not HAS_GROQ_KEY:
        parsed = _stub_summary(incident, docs)
        valid = _validate(parsed)
        return {**incident, "summary_text": parsed["summary"], "recommended_action": parsed["recommended_action"],
                "citation_doc_ids": ",".join(valid), "_summary_status": "stub" if valid else "needs_review"}

    parsed = _summarize_once(incident, docs)
    valid = _validate(parsed)
    if not valid:  # one stricter retry
        msgs = _build_prompt(incident, docs)
        msgs[1]["content"] += (f"\n\nIMPORTANT: You cited no valid policy. You MUST cite at least "
                               f"one of these ids: {', '.join(sorted(valid_ids))}. Do not output any other doc_id.")
        parsed = _parse_json_response(_chat(msgs))
        parsed["citations"] = list(dict.fromkeys(
            set(_extract_citations(parsed.get("summary", "")))
            | set(_extract_citations(parsed.get("recommended_action", "")))
            | set(parsed.get("citations", []) or [])))
        valid = _validate(parsed)
    if not valid:
        return {**incident, "summary_text": "[needs_review: no valid citation] " + parsed.get("summary", "")[:200],
                "recommended_action": parsed.get("recommended_action", ""), "citation_doc_ids": "",
                "_summary_status": "needs_review"}
    return {**incident, "summary_text": parsed.get("summary", "").strip(),
            "recommended_action": parsed.get("recommended_action", "").strip(),
            "citation_doc_ids": ",".join(valid), "_summary_status": "ok"}

# --- demo on the first incident ---
inc0 = incidents.iloc[0][["incident_id","incident_type","risk_band","risk_score","zone_id",
                          "linked_event_ids","linked_log_ids"]].to_dict()
out = summarize_incident(inc0)
print(f"mode={'LLM' if HAS_GROQ_KEY else 'STUB'}  status={out['_summary_status']}")
print("summary:", out["summary_text"])
print("action :", out["recommended_action"])
print("cites  :", out["citation_doc_ids"] or "-")

mode=LLM  status=ok
summary: A suspected unauthorized entry has occurred in restricted zone ZONE-D of SITE-003 with a risk score of 82. Surveillance anomaly confidence is 0.85. Immediate action is required to prevent further unauthorized access.
action : Dispatch on-site security to ZONE-D within 5 minutes to investigate and contain the incident.
cites  : KB-00001


# 🛡️ Governance & Agent

Policy gate, governance nodes, graph (with the plan-loop FIX), domain tools, prompts, copilot wiring.

## 12. Policy gate — the reuse linchpin

A small **predicate evaluator** (`gt/ge/lt/le/eq/ne/in/regex_match`) over a
`constraints` mapping. This is the *same mechanism* that enforced the donor
project's spend cap (`cost_estimate ≤ 500`) — here it enforces our
`risk_band_score ≥ 80` human-review gate. **One mechanism, two domains.**

Key rules from `policy.yaml`:
- `incident.escalate`: `side_effect`, `require_human`, constraint `risk_band_score >= 80`.
- `case.close`: `allow: false` — **hard block**, never auto-executed. A human closes a case.

In [13]:
_PREDICATES = {
    "gt": lambda v, o: v is not None and v > o,
    "ge": lambda v, o: v is not None and v >= o,
    "lt": lambda v, o: v is not None and v < o,
    "le": lambda v, o: v is not None and v <= o,
    "eq": lambda v, o: v == o,
    "ne": lambda v, o: v != o,
    "in": lambda v, o: v in o,
    "regex_match": lambda v, o: bool(re.search(o, str(v))) if v is not None else False,
}

def evaluate_constraints(args, constraints):
    # Return violation strings for any constraint that fails. A missing field
    # with constraints is itself a violation - fail closed.
    violations = []
    for field_name, preds in constraints.items():
        value = args.get(field_name) if args else None
        for pred_name, operand in preds.items():
            pred = _PREDICATES.get(pred_name)
            if pred is None:
                violations.append(f"unknown_predicate:{pred_name}"); continue
            if not pred(value, operand):
                violations.append(f"constraint_failed:{field_name}:{pred_name}:{operand}")
    return violations

@dataclass
class ActionRule:
    name: str; side_effect: bool; allow: bool; require_human: bool = False
    constraints: dict = field(default_factory=dict)
    require_fields: list = field(default_factory=list); block_reason: str = ""

@dataclass
class ReviewDecision:
    allow: bool; require_human: bool; violations: list = field(default_factory=list); reason: str = ""
    @property
    def route(self):
        if not self.allow: return "block"
        if self.require_human: return "require_human"
        return "allow"

@dataclass
class RedactionPattern:
    name: str; regex: str; replacement: str

@dataclass
class Policy:
    domain: str; actions: dict; redaction_patterns: list = field(default_factory=list)
    redaction_enabled: bool = True; intake: dict = field(default_factory=dict)
    retention: dict = field(default_factory=dict); raw: dict = field(default_factory=dict)

    @classmethod
    def from_dict(cls, data):
        actions = {}
        for a in data.get("actions", []):
            actions[a["name"]] = ActionRule(
                name=a["name"], side_effect=bool(a.get("side_effect", False)),
                allow=bool(a.get("allow", False)), require_human=bool(a.get("require_human", False)),
                constraints=a.get("constraints", {}) or {},
                require_fields=list(a.get("require_fields", []) or []),
                block_reason=a.get("block_reason", "") or "")
        pii = data.get("pii_redaction", {}) or {}
        patterns = [RedactionPattern(p["name"], p["regex"], p["replacement"]) for p in pii.get("patterns", []) or []]
        return cls(domain=data.get("domain", "unspecified"), actions=actions,
                   redaction_patterns=patterns, redaction_enabled=bool(pii.get("enabled", True)),
                   intake=data.get("intake", {}) or {}, retention=data.get("retention", {}) or {}, raw=data)

    def get_rule(self, action): return self.actions.get(action)

    def evaluate(self, action, args):
        rule = self.actions.get(action)
        if rule is None:
            return ReviewDecision(False, False, ["unknown_action"], f"Action '{action}' is not in the policy.")
        if not rule.allow:
            return ReviewDecision(False, rule.require_human, ["action_not_allowed"],
                                  rule.block_reason or f"Action '{action}' is not permitted.")
        violations = []
        for rf in rule.require_fields:
            if not args or args.get(rf) in (None, ""):
                violations.append(f"missing_required_field:{rf}")
        violations.extend(evaluate_constraints(args or {}, rule.constraints))
        if violations:
            return ReviewDecision(False, rule.require_human or rule.side_effect, violations,
                                  f"Action '{action}' failed policy checks: {violations}")
        return ReviewDecision(True, rule.require_human, [], f"Action '{action}' permitted by policy.")

# --- The SOC policy, inlined (same schema as the donor's property-management policy) ---
SOC_POLICY_DICT = {
  "domain": "security_operations",
  "pii_redaction": {"enabled": True, "patterns": [
    {"name": "email", "regex": r"[\w.+-]+@[\w-]+\.[\w.-]+", "replacement": "[EMAIL-REDACTED]"},
    {"name": "phone", "regex": r"\b\d{10}\b", "replacement": "[PHONE-REDACTED]"},
    {"name": "ssn",   "regex": r"\b\d{3}-\d{2}-\d{4}\b", "replacement": "[SSN-REDACTED]"},
  ]},
  "actions": [
    {"name": "incident.fuse",      "side_effect": False, "allow": True},
    {"name": "incident.score",     "side_effect": False, "allow": True},
    {"name": "sop.retrieve",       "side_effect": False, "allow": True},
    {"name": "incident.summarize", "side_effect": False, "allow": True},
    {"name": "incident.escalate",  "side_effect": True, "allow": True, "require_human": True,
     "constraints": {"risk_band_score": {"ge": 80}},
     "require_fields": ["incident_id", "risk_band_score"],
     "block_reason": "Escalation requires a critical-band incident and human approval."},
    {"name": "case.close", "side_effect": True, "allow": False, "require_human": True,
     "block_reason": "Case closure is outside agent authority; a human analyst must close it."},
  ],
}
policy = Policy.from_dict(SOC_POLICY_DICT)

# --- self-check: the two-domain linchpin in action ---
ok   = policy.evaluate("incident.escalate", {"incident_id": "INC-1", "risk_band_score": 85})
low  = policy.evaluate("incident.escalate", {"incident_id": "INC-1", "risk_band_score": 70})
shut = policy.evaluate("case.close", {"incident_id": "INC-1"})
print("escalate@85 ->", ok.route, "(allowed, needs human)")
print("escalate@70 ->", low.route, low.violations)
print("case.close  ->", shut.route, "(hard block)")
assert ok.route == "require_human" and low.route == "block" and shut.route == "block"
print("policy self-check OK")

escalate@85 -> require_human (allowed, needs human)
escalate@70 -> block ['constraint_failed:risk_band_score:ge:80']
case.close  -> block (hard block)
policy self-check OK


## 13. Governance state — dataclasses flowing through every node

Domain-agnostic. The only domain-specific bit is `domain_state`, an opaque dict
the domain layer fills (e.g. `incident_id` for the SOC copilot) — governance
nodes never read inside it.

In [14]:
@dataclass
class PlanStep:
    action: str; reason: str = ""; expected_side_effect: bool = False
    args: dict = field(default_factory=dict)

@dataclass
class ActionIntent:
    action: str; args: dict = field(default_factory=dict)
    side_effect: bool = False; cost_estimate: float | None = None

@dataclass
class ToolResult:
    tool: str; ok: bool; summary: str; payload: Any = None

@dataclass
class GovReviewDecision:
    allow: bool; require_human: bool; violations: list = field(default_factory=list); reason: str = ""
    @property
    def route(self):
        if not self.allow: return "block"
        if self.require_human: return "require_human"
        return "allow"

class AgentState(TypedDict, total=False):
    user_id: str; turn_id: str; messages: list; redacted_text: str
    plan: list; step_index: int
    current_action: Any; review: Any; tool_result: Any
    scratchpad_ref: str; domain_state: dict
    iteration: int; status: str

print("governance state ready")

governance state ready


## 14. Audit log + memory + PII redactor

**Audit log:** hash-chained JSONL — every line stores the hash of the previous
line, so tampering with a past entry breaks the chain and `verify_chain()`
catches it. Domain-agnostic.

**Memory:** a per-user SQLite scratchpad of recent turns (no vector store;
conversations are short, "recent N" is enough).

**PII redactor:** applies the policy's regex patterns. Domain-agnostic.

In [ ]:
GENESIS_HASH = hashlib.sha256(b"genesis-p6-audit").hexdigest()

class ChainBrokenError(Exception): 
    pass

def _canonical_json(obj): 
    return json.dumps(obj, sort_keys=True, separators=(",", ":"))

def _utc_now(): 
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

class AuditLogger:
    def __init__(self, path):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
        if self.path.exists() and self.path.stat().st_size > 0:
            lines = [l for l in self.path.read_text(encoding="utf-8").splitlines() if l.strip()]
            self._prev_hash = json.loads(lines[-1])["this_hash"]; self._seq = len(lines)
        else:
            self._prev_hash = GENESIS_HASH; self._seq = 0

    def _append(self, record):
        record["seq"] = self._seq; record["ts"] = _utc_now(); record["prev_hash"] = self._prev_hash
        this_hash = hashlib.sha256((self._prev_hash + _canonical_json(record)).encode("utf-8")).hexdigest()
        record["this_hash"] = this_hash
        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n"); f.flush(); os.fsync(f.fileno())
        self._seq += 1; self._prev_hash = this_hash
        return this_hash

    def log_call(self, *, turn_id, action, args, tool, result_summary, actor="worker"):
        return self._append({"turn_id": turn_id, "node": "worker_dispatch", "kind": "call",
                             "actor": actor, "action": action, "args": args or {},
                             "tool": tool, "result_summary": result_summary})
    def log_decision(self, *, turn_id, node, decision, rationale="", actor="system"):
        return self._append({"turn_id": turn_id, "node": node, "kind": "decision",
                             "actor": actor, "decision": decision, "rationale": rationale})
    def log_block(self, *, turn_id, action, args, violations, block_reason, actor="reviewer"):
        return self._append({"turn_id": turn_id, "node": "reviewer", "kind": "block",
                             "actor": actor, "action": action, "args": args or {},
                             "violations": violations, "block_reason": block_reason})
    def log_human_approval(self, *, turn_id, action, approver, granted, note=""):
        return self._append({"turn_id": turn_id, "node": "human_approval", "kind": "human_approval",
                             "actor": approver, "action": action, "granted": granted, "note": note})

    def verify_chain(self):
        if not self.path.exists(): return True
        prev = GENESIS_HASH
        for line in self.path.read_text(encoding="utf-8").splitlines():
            if not line.strip(): continue
            record = json.loads(line); this_hash = record.pop("this_hash")
            expected = hashlib.sha256((prev + _canonical_json(record)).encode("utf-8")).hexdigest()
            if expected != this_hash:
                raise ChainBrokenError(f"chain broken at seq={record.get('seq')}")
            prev = this_hash
        return True
    def read_all(self):
        return [json.loads(l) for l in self.path.read_text(encoding="utf-8").splitlines() if l.strip()]

class SessionScratchpad:
    def __init__(self, path):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
        with sqlite3.connect(self.path) as conn:
            conn.execute('''CREATE TABLE IF NOT EXISTS scratchpad (
                user_id TEXT NOT NULL, turn_id TEXT NOT NULL, seq INTEGER NOT NULL,
                ts TEXT NOT NULL, kind TEXT NOT NULL, content TEXT NOT NULL,
                PRIMARY KEY (user_id, turn_id, seq))''')
            conn.execute("CREATE INDEX IF NOT EXISTS idx_scratch_user_ts ON scratchpad(user_id, ts DESC)")
    def append(self, user_id, turn_id, kind, content):
        with sqlite3.connect(self.path) as conn:
            conn.execute("INSERT INTO scratchpad (user_id, turn_id, seq, ts, kind, content) "
                         "SELECT ?, ?, COALESCE(MAX(seq), -1) + 1, datetime('now'), ?, ? "
                         "FROM scratchpad WHERE user_id = ? AND turn_id = ?",
                         (user_id, turn_id, kind, content, user_id, turn_id))
    def recent(self, user_id, n=5):
        with sqlite3.connect(self.path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute("SELECT kind, content, ts FROM scratchpad WHERE user_id = ? "
                                "ORDER BY ts DESC, seq DESC LIMIT ?", (user_id, n)).fetchall()
        return [dict(r) for r in rows][::-1]
    def clear(self):
        with sqlite3.connect(self.path) as conn: conn.execute("DELETE FROM scratchpad")

def redact(text, p: Policy):
    if not p.redaction_enabled or not text: return text
    out = text
    for pat in p.redaction_patterns:
        out = re.sub(pat.regex, pat.replacement, out)
    return out

# self-check
demo_audit = AuditLogger(AGENT_DATA_DIR / "_demo_audit.jsonl")
demo_audit.log_decision(turn_id="t1", node="planner", decision="plan_issued")
demo_audit.log_block(turn_id="t1", action="case.close", args={}, violations=["action_not_allowed"],
                     block_reason="outside authority")
assert demo_audit.verify_chain() is True
print("audit chain OK; redact demo:", redact("contact analyst@corp.com re INC-1", policy))
Path(AGENT_DATA_DIR / "_demo_audit.jsonl").unlink(missing_ok=True)

audit chain OK; redact demo: contact [EMAIL-REDACTED] re INC-1


## 15. Governance nodes + routers + parsers

Each node is `node(state) -> state_update`. The LLM-touching nodes go through
one helper, `call_llm`, which **falls back to a deterministic stub when
`llm is None`** — so the whole graph runs end-to-end without a model.

**The FIX lives in `route_after_dispatch`:** the loop target is `"worker"`,
not `"reviewer"`. Routing back to the worker re-loads `plan[step_index]` into
`current_action` each iteration *before* the reviewer gates it. (See the next
section's graph diagram for why this matters.)

In [16]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

def call_llm(llm, system, user):
    if llm is None:
        return json.dumps({"action": "noop", "args": {}, "summary": "[stub] no LLM configured"})
    msgs = [SystemMessage(content=system), HumanMessage(content=user)]
    resp = llm.invoke(msgs)
    return resp.content if isinstance(resp, AIMessage) else str(resp)

def make_planner_node(llm, prompts, memory, audit):
    def planner(state):
        if state.get("plan"):  # pre-injected plan (stub / scripted runs)
            audit.log_decision(turn_id=state["turn_id"], node="planner",
                               decision="plan_pre_injected", rationale="plan supplied by intake")
            return {"step_index": 0, "iteration": 0, "status": "executing"}
        user_text = state.get("redacted_text", "")
        prior = memory.recent(state["user_id"], n=3)
        prior_block = "\n".join(f"- {p['kind']}: {p['content']}" for p in prior) or "(none)"
        sys_prompt = prompts.planner_system + "\n\n<prior_turns>\n" + prior_block + "\n</prior_turns>"
        raw = call_llm(llm, sys_prompt, user_text)
        plan = parse_plan(raw)
        audit.log_decision(turn_id=state["turn_id"], node="planner", decision="plan_issued", rationale=raw[:200])
        memory.append(state["user_id"], state["turn_id"], "plan", raw[:500])
        return {"plan": plan, "step_index": 0, "iteration": 0, "status": "executing",
                "messages": state.get("messages", []) + [AIMessage(content=raw)]}
    return planner

def make_worker_node(llm, prompts, tool_specs):
    def worker(state):
        if not state.get("plan"):
            return {"current_action": None, "status": "done"}
        step: PlanStep = state["plan"][state["step_index"]]
        if llm is None:  # stub: build intent straight from the plan step
            return {"current_action": ActionIntent(action=step.action, args=dict(step.args),
                                                   side_effect=step.expected_side_effect)}
        raw = call_llm(llm, prompts.worker_system,
                       f"Action to perform: {step.action}\nReason: {step.reason}\n"
                       f"Available tools: {', '.join(t.name for t in tool_specs)}")
        return {"current_action": parse_action_intent(raw, step)}
    return worker

def make_reviewer_node(policy, audit):
    def reviewer(state):
        intent = state.get("current_action")
        if intent is None:
            return {"review": GovReviewDecision(False, False, ["no_action"], "no plan produced")}
        decision = policy.evaluate(intent.action, intent.args)
        rd = GovReviewDecision(decision.allow, decision.require_human, decision.violations, decision.reason)
        if rd.allow:
            audit.log_decision(turn_id=state["turn_id"], node="reviewer", decision="allow", rationale=rd.reason)
        else:
            audit.log_block(turn_id=state["turn_id"], action=intent.action, args=intent.args,
                            violations=rd.violations, block_reason=rd.reason)
        return {"review": rd}
    return reviewer

def make_worker_dispatch_node(tool_registry, audit):
    def worker_dispatch(state):
        intent = state["current_action"]
        tool_fn = tool_registry.get(intent.action)
        if tool_fn is None:
            result = ToolResult(intent.action, False, f"no tool registered for {intent.action}")
        else:
            try:
                payload = tool_fn(**intent.args)
                result = ToolResult(intent.action, True, str(payload)[:300], payload)
            except Exception as e:
                result = ToolResult(intent.action, False, f"tool error: {e}")
        audit.log_call(turn_id=state["turn_id"], action=intent.action, args=intent.args,
                       tool=result.tool, result_summary=result.summary)
        return {"tool_result": result, "step_index": state["step_index"] + 1}
    return worker_dispatch

def make_summarizer_node(llm, prompts, memory, audit):
    def summarizer(state):
        review, tr = state.get("review"), state.get("tool_result")
        if review and not review.allow:
            action_name = state["current_action"].action if state.get("current_action") else "(no action)"
            recap = f"Action '{action_name}' was blocked: {review.reason}"
            summary_text = call_llm(llm, prompts.summarizer_system, recap) if llm else recap
            memory.append(state["user_id"], state["turn_id"], "block", recap)
        elif tr:
            recap = f"Action '{tr.tool}' executed: {tr.summary}"
            summary_text = call_llm(llm, prompts.summarizer_system, recap) if llm else recap
            memory.append(state["user_id"], state["turn_id"], "tool", recap)
        else:
            recap = "turn complete"; summary_text = recap
        memory.append(state["user_id"], state["turn_id"], "summary", summary_text[:500])
        audit.log_decision(turn_id=state["turn_id"], node="summarizer",
                           decision="turn_complete", rationale=summary_text[:200])
        return {"status": "done"}
    return summarizer

def make_human_approval_node(audit):
    def human_approval(state):
        intent = state["current_action"]; review = state.get("review")
        if os.environ.get("COPILOT_HUMAN_GATE") == "1":
            from langgraph.types import interrupt
            decision = interrupt({"action": intent.action, "args": intent.args,
                                  "reason": getattr(review, "reason", "") if review else "",
                                  "turn_id": state["turn_id"]})
            granted = bool(decision)
            audit.log_human_approval(turn_id=state["turn_id"], action=intent.action,
                                     approver="human_via_interrupt", granted=granted,
                                     note="via langgraph interrupt (COPILOT_HUMAN_GATE=1)")
            if review: review.allow = granted; review.require_human = False
            return {"review": review}
        audit.log_human_approval(turn_id=state["turn_id"], action=intent.action,
                                 approver="notebook_operator", granted=True, note="auto-approved in demo mode")
        if review: review.allow = True
        return {"review": review}
    return human_approval

# --- Routers (module-level fns so the graph serializes) ---
def route_after_review(state): return state["review"].route

def route_after_dispatch(state):
    # FIX: return "worker", not "reviewer". Re-loads plan[step_index] each loop
    # before the gate; the non-bypassable gate (worker->reviewer->dispatch) is
    # preserved per iteration. See graph_builder for the full rationale.
    if state["step_index"] >= len(state["plan"]): return "summarizer"
    return "worker"

# --- Parsers (forgiving — LLM output is messy) ---
def parse_plan(raw):
    text = raw.strip()
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 3:
            text = parts[1]
            if text.lstrip().startswith(("json", "JSON")):
                text = text.split("\n", 1)[1] if "\n" in text else text
            text = text.strip()
    try:
        items = json.loads(text)
        if isinstance(items, list):
            return [PlanStep(i["action"], i.get("reason", ""), i.get("expected_side_effect", False))
                    for i in items if isinstance(i, dict) and "action" in i]
        if isinstance(items, dict) and "action" in items:
            return [PlanStep(items["action"], items.get("reason", ""), items.get("expected_side_effect", False))]
    except (json.JSONDecodeError, TypeError, KeyError): pass
    return [PlanStep("unknown", raw[:200])]

def parse_action_intent(raw, step):
    try:
        data = json.loads(raw)
        if isinstance(data, dict) and "action" in data:
            return ActionIntent(data["action"], data.get("args", {}) or {},
                                bool(data.get("side_effect", step.expected_side_effect)),
                                data.get("cost_estimate"))
    except (json.JSONDecodeError, TypeError): pass
    return ActionIntent(step.action, side_effect=step.expected_side_effect)

print("governance nodes + routers + parsers ready")

governance nodes + routers + parsers ready


## 16. Graph builder — with the multi-step plan-loop FIX

```
START -> ingest -> planner -> worker
worker -> reviewer                                   (always, before any tool runs)
reviewer -> {allow: worker_dispatch, require_human: human_approval, block: summarizer}
worker_dispatch -> {more steps: worker, done: summarizer}   <-- THE FIX (was -> reviewer)
human_approval -> worker_dispatch                     (after a human approves)
summarizer -> END
```

**Why the loop goes `dispatch -> worker`, not `dispatch -> reviewer`:** the
`worker` is the only node that loads `plan[step_index]` into `current_action`.
If the loop skipped it, the worker would run **once** and every later iteration
would re-review and re-dispatch **step 0's action** while `step_index` advanced
unused — a 5-step plan would silently re-run step 0 five times. Single-step
plans hid this in the donor; P7's multi-step plans exposed it. Routing back to
the worker re-loads the next step before the gate; the gate is preserved every
iteration, so the tool still never runs unreviewed.

In [17]:
from langgraph.graph import END, START, StateGraph

def build_graph(*, policy, tool_registry, tool_specs, audit, llm, memory,
                intake_fn, prompts, checkpointer=None):
    graph = StateGraph(AgentState)
    graph.add_node("ingest", intake_fn)
    graph.add_node("planner", make_planner_node(llm, prompts, memory, audit))
    graph.add_node("worker", make_worker_node(llm, prompts, tool_specs))
    graph.add_node("reviewer", make_reviewer_node(policy, audit))
    graph.add_node("worker_dispatch", make_worker_dispatch_node(tool_registry, audit))
    graph.add_node("summarizer", make_summarizer_node(llm, prompts, memory, audit))
    graph.add_node("human_approval", make_human_approval_node(audit))

    graph.add_edge(START, "ingest")
    graph.add_edge("ingest", "planner")
    graph.add_edge("planner", "worker")
    # The worker NEVER calls a tool directly -> always reviewer first. This edge
    # is the non-bypassable gate: the only way to reach worker_dispatch is via
    # the reviewer's "allow" route.
    graph.add_edge("worker", "reviewer")
    graph.add_edge("human_approval", "worker_dispatch")
    graph.add_edge("summarizer", END)

    graph.add_conditional_edges("reviewer", route_after_review,
        {"allow": "worker_dispatch", "require_human": "human_approval", "block": "summarizer"})
    # THE FIX: loop back to "worker" (re-loads next plan step), not "reviewer".
    graph.add_conditional_edges("worker_dispatch", route_after_dispatch,
        {"summarizer": "summarizer", "worker": "worker"})
    return graph.compile(checkpointer=checkpointer)

print("graph builder ready (with dispatch -> worker FIX)")

graph builder ready (with dispatch -> worker FIX)


## 17. Domain tools — the 6 SOC actions

Each tool is a plain function keyed by the dotted action name (matching the
policy). `worker_dispatch` calls these **after** the reviewer approves. Tools
are thin — they delegate to the fusion / RAG / summarizer already built above.

`case.close` **raises if it ever runs** — the policy hard-blocks it, so
reaching dispatch means the gate failed (a safety violation surfaced loudly).

In [18]:
def _load_incidents_df():
    return read_parquet("incidents")

def _incident_row(incident_id):
    df = _load_incidents_df()
    row = df[df["incident_id"] == incident_id]
    if row.empty: raise ValueError(f"incident {incident_id} not found")
    return row.iloc[0].to_dict()

def incident_fuse(incident_id=""):
    if not incident_id:
        df = _load_incidents_df()
        return {"count": len(df), "incident_ids": df["incident_id"].tolist()}
    row = _incident_row(incident_id)
    return {"incident_id": row["incident_id"], "incident_type": row["incident_type"],
            "risk_score": int(row["risk_score"]), "risk_band": row["risk_band"],
            "zone_id": row["zone_id"], "linked_event_ids": row["linked_event_ids"],
            "linked_log_ids": row["linked_log_ids"]}

def incident_score(incident_id):
    row = _incident_row(incident_id)
    return {"incident_id": row["incident_id"], "risk_score": int(row["risk_score"]),
            "risk_band": row["risk_band"], "human_review_required": bool(row["human_review_required"])}

def _query_text(row):
    return (f"{row['incident_type']} in zone {row['zone_id']}; "
            f"events {row.get('linked_event_ids', '')}; logs {row.get('linked_log_ids', '')}")

def sop_retrieve(incident_id, k=3):
    row = _incident_row(incident_id)
    docs = retrieve_for_incident(row["incident_type"], _query_text(row), k=k)
    return {"incident_id": incident_id, "incident_type": row["incident_type"],
            "docs": [{"doc_id": d["doc_id"], "title": d["title"], "category": d["category"],
                      "score": round(d["score"], 3)} for d in docs]}

def incident_summarize(incident_id):
    row = _incident_row(incident_id)
    out = summarize_incident({"incident_id": row["incident_id"], "incident_type": row["incident_type"],
                              "risk_band": row["risk_band"], "risk_score": int(row["risk_score"]),
                              "zone_id": row["zone_id"], "linked_event_ids": row["linked_event_ids"],
                              "linked_log_ids": row["linked_log_ids"]})
    return {"incident_id": out["incident_id"], "summary_text": out["summary_text"],
            "recommended_action": out["recommended_action"],
            "citation_doc_ids": out["citation_doc_ids"], "status": out.get("_summary_status", "ok")}

def incident_escalate(incident_id, risk_band_score):
    row = _incident_row(incident_id)
    return {"incident_id": incident_id, "risk_band_score": risk_band_score,
            "risk_band": row["risk_band"], "escalated": risk_band_score >= 75,
            "action": "page_duty_manager",
            "note": "Escalation recorded; awaiting human approval (demo auto-approves)."}

def case_close(incident_id):
    raise RuntimeError("case.close reached dispatch - the policy gate failed to block it. "
                       "Case closure must be a human act.")

# --- kwarg filter so a free model's invented arg names don't crash dispatch ---
def _filter_kwargs(fn, kwargs):
    sig = inspect.signature(fn); params = sig.parameters
    if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()): return kwargs
    return {k: v for k, v in kwargs.items() if k in params}

def _wrap(fn):
    def wrapped(**kwargs): return fn(**_filter_kwargs(fn, kwargs))
    wrapped.__name__ = fn.__name__
    return wrapped

TOOL_REGISTRY = {
    "incident.fuse": _wrap(incident_fuse),
    "incident.score": _wrap(incident_score),
    "sop.retrieve": _wrap(sop_retrieve),
    "incident.summarize": _wrap(incident_summarize),
    "incident.escalate": _wrap(incident_escalate),
    "case.close": _wrap(case_close),
}

# quick self-check
_iid = incidents.iloc[0]["incident_id"]
print("fuse  :", incident_fuse(_iid)["incident_type"], incident_fuse(_iid)["risk_band"])
print("score :", incident_score(_iid))
print("retrieve:", [d["doc_id"] for d in sop_retrieve(_iid, k=2)["docs"]])

fuse  : suspected_unauthorized_entry critical
score : {'incident_id': 'INC-000001', 'risk_score': 82, 'risk_band': 'critical', 'human_review_required': True}
retrieve: ['KB-00001', 'KB-00002']


## 18. Prompts + intake node

SOC analyst voice. The governance nodes don't read prompts — they just pass
them to the LLM, so swapping them changes behavior without touching the graph.

The **intake node** is event-driven (not OCR-driven like the donor): the caller
puts an `incident_id` in `domain_state`, intake reads that incident, redacts
PII, and surfaces a short text summary as `redacted_text` for the planner.

In [19]:
@dataclass
class SOCPrompts:
    planner_system: str = (
        "You are a Security Operations Center copilot assisting an SOC analyst. "
        "Read the incident triage request and produce a short plan (1-4 steps) "
        "as a JSON list. Each step is an object with keys: action, reason, "
        "expected_side_effect. Allowed actions: incident.fuse, incident.score, "
        "sop.retrieve, incident.summarize, incident.escalate, case.close. "
        "Never auto-close a case. Output only the JSON list, no prose.")
    worker_system: str = (
        "You fill in the arguments for one SOC action. Output a single JSON "
        "object with keys: action, args. Use ONLY these actions and EXACT "
        "argument names:\n"
        "- incident.fuse: {incident_id}\n"
        "- incident.score: {incident_id}\n"
        "- sop.retrieve: {incident_id}\n"
        "- incident.summarize: {incident_id}\n"
        "- incident.escalate: {incident_id, risk_band_score}\n"
        "- case.close: {incident_id}\n"
        "incident_id is a string like 'INC-000001'. risk_band_score is an "
        "integer 0-100. Output only the JSON object, no prose.")
    summarizer_system: str = (
        "Write one or two plain sentences for the SOC analyst describing what "
        "just happened. If an action was blocked, say so and give the reason. "
        "Do not invent details.")

DEFAULT_PROMPTS = SOCPrompts()

def _incident_text(incident_id):
    df = read_parquet("incidents")
    row = df[df["incident_id"] == incident_id]
    if row.empty: raise ValueError(f"incident {incident_id} not found - run fusion first.")
    r = row.iloc[0]
    return (f"Incident {r['incident_id']} ({r['incident_type']}), "
            f"risk_band={r['risk_band']}, risk_score={r['risk_score']}, zone={r['zone_id']}. "
            f"Linked events: {r['linked_event_ids'] or 'none'}; "
            f"linked logs: {r['linked_log_ids'] or 'none'}.")

def make_intake_node(policy, audit, llm=None, ocr_threshold=65):
    def intake(state):
        user_text = state.get("redacted_text", "")
        if not user_text:
            incident_id = state.get("domain_state", {}).get("incident_id")
            if not incident_id:
                audit.log_decision(turn_id=state.get("turn_id", ""), node="ingest",
                                   decision="intake_no_incident",
                                   rationale="domain_state has no incident_id")
                return {"redacted_text": "", "messages": state.get("messages", [])}
            user_text = _incident_text(incident_id)
            audit.log_decision(turn_id=state.get("turn_id", ""), node="ingest",
                               decision="intake_staged_incident",
                               rationale=f"incident_id={incident_id}")
        redacted = redact(user_text, policy)
        return {"redacted_text": redacted, "messages": state.get("messages", [])}
    return intake

print("prompts + intake node ready")

prompts + intake node ready


## 19. Copilot agent — wire it all together

`build_system` assembles policy + audit + memory + tools + (optional) LLM and
compiles the graph. `run_incident` runs the copilot on one incident.

**Stub mode (default, no `GROQ_API_KEY`):** a scripted plan is injected so the
run is deterministic without an LLM. The plan is the canonical SOC triage:
`fuse → score → retrieve → summarize`, plus `escalate` if the band is critical.
**`--llm` mode:** the planner generates the plan from the staged incident text
via Groq (the real agent path).

`GroqChat` is a minimal LangChain-compatible adapter over `requests` — exposes
only `.invoke(messages) -> AIMessage`, which is all the governance nodes call.
No OpenAI SDK, no LangChain wrapper.

In [20]:
@dataclass
class BuiltSystem:
    graph: object; audit: AuditLogger; memory: SessionScratchpad
    policy: Policy; llm: object

class GroqChat:
    # Minimal LangChain-compatible chat model over Groq's OpenAI-compatible endpoint.
    def __init__(self, model=None, temperature=0.2, max_retries=3, timeout=60, api_key=None):
        self.api_key = api_key or _api_key()
        self.model = model or os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant")
        self.url = os.environ.get("GROQ_BASE_URL", "https://api.groq.com/openai/v1") + "/chat/completions"
        self.temperature, self.max_retries, self.timeout = temperature, max_retries, timeout

    def _to_role(self, m):
        if hasattr(m, "type") and hasattr(m, "content"):
            role = "assistant" if m.type == "ai" else m.type
            if role == "human": role = "user"
            return {"role": role, "content": m.content}
        if isinstance(m, dict): return {"role": m["role"], "content": m["content"]}
        return {"role": "user", "content": str(m)}

    def invoke(self, messages, **_kw):
        from langchain_core.messages import AIMessage
        body = {"model": self.model, "temperature": self.temperature,
                "messages": [self._to_role(m) for m in messages]}
        headers = {"Authorization": f"Bearer {self.api_key}", "Content-Type": "application/json"}
        last = None
        for attempt in range(self.max_retries):
            try:
                resp = requests.post(self.url, headers=headers, json=body, timeout=self.timeout)
            except requests.RequestException as e:
                last = e; time.sleep(2 ** attempt); continue
            if resp.status_code == 200:
                return AIMessage(content=resp.json()["choices"][0]["message"]["content"])
            if resp.status_code in (429, 500, 502, 503, 504):
                last = RuntimeError(f"Groq {resp.status_code}: {resp.text[:200]}")
                time.sleep(2 ** attempt); continue
            raise RuntimeError(f"Groq {resp.status_code}: {resp.text[:300]}")
        raise RuntimeError(f"Groq failed after {self.max_retries} retries: {last}")

def build_system(use_llm=False, api_key=None):
    AGENT_DATA_DIR.mkdir(parents=True, exist_ok=True)
    pol = Policy.from_dict(SOC_POLICY_DICT)
    audit = AuditLogger(AGENT_DATA_DIR / "audit.jsonl")
    memory = SessionScratchpad(AGENT_DATA_DIR / "scratchpad.db")
    llm = GroqChat(api_key=api_key) if use_llm else None
    intake = make_intake_node(pol, audit, llm=llm)
    graph = build_graph(policy=pol, tool_registry=TOOL_REGISTRY, tool_specs=[],
                        audit=audit, llm=llm, memory=memory, intake_fn=intake, prompts=DEFAULT_PROMPTS)
    return BuiltSystem(graph=graph, audit=audit, memory=memory, policy=pol, llm=llm)

def _plan_for(incident):
    # Canonical SOC triage plan: fuse -> score -> retrieve -> summarize,
    # then escalate if the band is critical. Deterministic; works in stub mode.
    steps = [
        PlanStep("incident.fuse", "read the fused incident", False, {"incident_id": incident["incident_id"]}),
        PlanStep("incident.score", "confirm risk band", False, {"incident_id": incident["incident_id"]}),
        PlanStep("sop.retrieve", "fetch relevant policy", False, {"incident_id": incident["incident_id"]}),
        PlanStep("incident.summarize", "analyst-facing summary", False, {"incident_id": incident["incident_id"]}),
    ]
    if incident["risk_band"] == "critical":
        steps.append(PlanStep("incident.escalate", "critical band -> escalate", True,
                              {"incident_id": incident["incident_id"], "risk_band_score": int(incident["risk_score"])}))
    return steps

def _intake_with_plan(incident, base_intake):
    plan = _plan_for(incident)
    def intake(state):
        out = base_intake(state); out["plan"] = plan; return out
    return intake

def run_incident(incident_id, use_llm=False, api_key=None):
    df = read_parquet("incidents")
    row = df[df["incident_id"] == incident_id]
    if row.empty: raise ValueError(f"incident {incident_id} not found")
    incident = row.iloc[0].to_dict(); incident["risk_score"] = int(incident["risk_score"])
    sys_ = build_system(use_llm=use_llm, api_key=api_key)
    if use_llm:
        intake = make_intake_node(sys_.policy, sys_.audit, llm=sys_.llm)
    else:
        intake = _intake_with_plan(incident, make_intake_node(sys_.policy, sys_.audit, llm=sys_.llm))
    graph = build_graph(policy=sys_.policy, tool_registry=TOOL_REGISTRY, tool_specs=[],
                        audit=sys_.audit, llm=sys_.llm, memory=sys_.memory,
                        intake_fn=intake, prompts=DEFAULT_PROMPTS)
    init = {"user_id": "analyst-1", "turn_id": f"turn-{incident_id}",
            "messages": [], "domain_state": {"incident_id": incident_id}}
    return graph.invoke(init, config={"recursion_limit": 25})

print("copilot agent ready. HAS_GROQ_KEY =", HAS_GROQ_KEY)

copilot agent ready. HAS_GROQ_KEY = True


# 🚀 Run

Run an incident end to end and inspect the audit trail.

## 20. Run an incident end to end + inspect the audit trail

Pick an incident and run the full copilot. In stub mode (no key) this is fully
deterministic. We pick a **critical-band** incident if one exists so the
escalate step + human-approval gate actually fire; otherwise the first
incident. Then we dump the hash-chained audit trail to see every reviewed
decision, tool call, block, and approval in order.

In [21]:
# Pick a critical incident if we have one, else the first row.
crit = incidents[incidents["risk_band"] == "critical"]
target = crit.iloc[0]["incident_id"] if not crit.empty else incidents.iloc[0]["incident_id"]
target_row = incidents[incidents["incident_id"] == target].iloc[0]
print(f"Running copilot on {target}  type={target_row['incident_type']} "
      f"band={target_row['risk_band']} score={target_row['risk_score']}")

final = run_incident(target, use_llm=HAS_GROQ_KEY)
print(f"\n=== {target} | final status={final.get('status')} ===")
tr = final.get("tool_result")
if tr:
    print(f"last tool: {tr.tool} ok={tr.ok}")
    print(f"  summary: {tr.summary}")
rev = final.get("review")
if rev and not rev.allow:
    print(f"BLOCKED: {rev.reason}")

Running copilot on INC-000001  type=suspected_unauthorized_entry band=critical score=82

=== INC-000001 | final status=done ===
last tool: incident.fuse ok=True
  summary: {'incident_id': 'INC-000001', 'incident_type': 'suspected_unauthorized_entry', 'risk_score': 82, 'risk_band': 'critical', 'zone_id': 'SITE-003::ZONE-D', 'linked_event_ids': 'EVT-000038', 'linked_log_ids': ''}


In [22]:
# --- Inspect the hash-chained audit trail for this turn ----------------------
audit = AuditLogger(AGENT_DATA_DIR / "audit.jsonl")
print("chain valid:", audit.verify_chain())
print(f"{'seq':>3} {'kind':16s} {'node':16s} {'action':22s} detail")
print("-" * 90)
for r in audit.read_all():
    if r["turn_id"] != f"turn-{target}": continue
    kind, node = r["kind"], r["node"]
    if kind == "decision":
        detail = f"{r['decision']}: {r['rationale'][:50]}"
    elif kind == "call":
        detail = f"tool={r['tool']} -> {r['result_summary'][:50]}"
    elif kind == "block":
        detail = f"violations={r['violations']} reason={r['block_reason'][:40]}"
    elif kind == "human_approval":
        detail = f"granted={r['granted']} by={r['actor']} note={r['note'][:40]}"
    else:
        detail = json.dumps({k: v for k, v in r.items() if k not in ('seq','ts','prev_hash','this_hash','turn_id')})[:60]
    print(f"{r['seq']:>3} {kind:16s} {node:16s} {r.get('action',''):22s} {detail}")

print("\nWhat just happened (stub mode = deterministic):")
print("  ingest   -> staged the incident + redacted PII")
print("  planner  -> accepted the injected plan (fuse/score/retrieve/summarize[/escalate])")
print("  each step: worker loads plan[i] -> reviewer gates it -> dispatch runs the tool")
print("  the loop goes dispatch -> WORKER (the FIX), so step_index advances each iteration")
print("  escalate (if critical) -> reviewer routes to human_approval (risk_band_score >= 80)")
print("  case.close would be hard-blocked by the reviewer -> straight to summarizer")

chain valid: True
seq kind             node             action                 detail
------------------------------------------------------------------------------------------
  0 decision         ingest                                  intake_staged_incident: incident_id=INC-000001
  1 decision         planner                                 plan_issued: [
  {
    "action": "incident.summarize",
    "rea
  2 decision         reviewer                                allow: Action 'incident.summarize' permitted by policy.
  3 call             worker_dispatch  incident.summarize     tool=incident.summarize -> {'incident_id': 'INC-000001', 'summary_text': 'A s
  4 decision         reviewer                                allow: Action 'incident.fuse' permitted by policy.
  5 call             worker_dispatch  incident.fuse          tool=incident.fuse -> {'incident_id': 'INC-000001', 'incident_type': 'su
  6 decision         reviewer                                allow: Action 'sop.retrieve

## Done

You've now traced the whole SOC copilot in one notebook, start to end:

1. **Data** (sites/zones/devices/users → surveillance events → access logs)
2. **Fusion** (4 rule detectors → risk scoring → `incidents.parquet`)
3. **RAG** (5-doc KB → Chroma → MMR + category routing)
4. **Summarizer** (Groq + citation guard, with a deterministic stub fallback)
5. **Governance-gated agent** (planner → worker → reviewer → dispatch loop, with
   the `dispatch → worker` FIX; human approval on critical escalation;
   `case.close` hard-blocked)

**Two thresholds, two jobs:** `0.85` is the *confidence* trigger for the
intrusion rule (detection stage); `80` is the *risk* gate that forces human
approval before escalation (scoring/governance stage). They never compete.

**To run with the real LLM:** set `GROQ_API_KEY` (in `project_07_final_synthesis/.env`
or your shell) and re-run from Section 19 with `use_llm=True`. The graph,
policy gate, and citation guard are identical — only the planner stops using
the injected script and generates the plan from the staged incident text.